# è fortemente consigliato runnare il notebook su colab

# Preliminari

In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
import torch
import json
import copy
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

# Parametri

In [4]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model_name = "unsloth/Llama-3.2-3B-Instruct"
DATA_FILE_NAME = "data_training.json"
CHAT_TEMPLATE = "llama-3.1"

# Load model and tokenizer

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Data preparation

In [10]:
# Carichiamo il nostro file JSON
with open(DATA_FILE_NAME, "r") as f:
    raw_json_data = json.load(f)

# I dati sono in queso formato:
"""
{
    "conversations": [
        {"messages": []},  # Primo referto
        {"messages": []},  # Secondo referto
        ...
    ]
}
"""

'\n{\n    "conversations": [\n        {"messages": []},  # Primo referto\n        {"messages": []},  # Secondo referto\n        ...\n    ]\n}\n'

## Explanation about chat templates

We now use the `Llama-3.1` format for conversation style finetunes. We convert it to HuggingFace's normal multiturn format `("role", "content")` Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

In [7]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = CHAT_TEMPLATE,
)

In [8]:
# Example: how chat template works
messages = [
    {'role': 'system', 'content': 'You are a helpful, respectful and honest assistant.'},
    {'role': 'user', 'content': 'Who are you?'},
    {'role': 'assistant', 'content': 'I am a large language model trained by Meta.'}
]
print(tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|>


In [9]:
# NON VA BENE AGGIUNGERE add_generation_prompt = True, infatti:
print(tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Bisogna fare in modo che assistant_content sia una stringa!!!

In [11]:
print(f"Tipo dato = {type(raw_json_data['conversations'][0]['messages'][2]['content'])}")
display(raw_json_data['conversations'][0]['messages'][2]['content'])

Tipo dato = <class 'dict'>


{'morfologia': 'solido_polipoide',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 49.0,
 'distanza_oai': 50.0,
 'posizione': 'medio',
 'carcinosi_peritoneale': None,
 'lesioni_ossee': None,
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'si',
 'linfonodi_sospetti': 4.0,
 'sedi_locoregionali': ['mesorettali', 'sacrali', 'mesenterici_inferiori'],
 'sedi_non_locoregionali': [],
 'depositi_tumorali': 'si',
 'numero_depositi': 0.0,
 'emvi_esteso': 'si'}

In [12]:
print(f"Tipo dato = {type(json.dumps(raw_json_data['conversations'][0]['messages'][2]['content']))}")
print(json.dumps(raw_json_data['conversations'][0]['messages'][2]['content'], indent=4))

Tipo dato = <class 'str'>
{
    "morfologia": "solido_polipoide",
    "spessore_parietale": null,
    "estensione_cranio_caudale": 49.0,
    "distanza_oai": 50.0,
    "posizione": "medio",
    "carcinosi_peritoneale": null,
    "lesioni_ossee": null,
    "riflessione_peritoneale_anteriore": "sotto",
    "infiltrazione_tessuto_adiposo": "si_5mm_plus",
    "infiltrazione_sfinteri": "no",
    "infiltrazione_organi_extra": "no",
    "infiltrazione_organi_dettagli": null,
    "coinvolgimento_riflessione_peritoneale": "no",
    "coinvolgimento_fascia_mesorettale": "si",
    "linfonodi_sospetti": 4.0,
    "sedi_locoregionali": [
        "mesorettali",
        "sacrali",
        "mesenterici_inferiori"
    ],
    "sedi_non_locoregionali": [],
    "depositi_tumorali": "si",
    "numero_depositi": 0.0,
    "emvi_esteso": "si"
}


Trasformiamo tutti i contenuti dell'astistente in stringhe.

## Creazione dataset corretto

In [13]:
training_data = copy.deepcopy(raw_json_data)

for conversation in training_data['conversations']:
    for message in conversation['messages']:
        if message['role'] == 'assistant':
            message['content'] = json.dumps(message['content'])

In [14]:
# Notare ora la differenza
display(training_data['conversations'][0]['messages'][2]['content'])
print(100*'#')
display(raw_json_data['conversations'][0]['messages'][2]['content'])

'{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 49.0, "distanza_oai": 50.0, "posizione": "medio", "carcinosi_peritoneale": null, "lesioni_ossee": null, "riflessione_peritoneale_anteriore": "sotto", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "sedi_locoregionali": ["mesorettali", "sacrali", "mesenterici_inferiori"], "sedi_non_locoregionali": [], "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}'

####################################################################################################


{'morfologia': 'solido_polipoide',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 49.0,
 'distanza_oai': 50.0,
 'posizione': 'medio',
 'carcinosi_peritoneale': None,
 'lesioni_ossee': None,
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'si',
 'linfonodi_sospetti': 4.0,
 'sedi_locoregionali': ['mesorettali', 'sacrali', 'mesenterici_inferiori'],
 'sedi_non_locoregionali': [],
 'depositi_tumorali': 'si',
 'numero_depositi': 0.0,
 'emvi_esteso': 'si'}

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [15]:
# Esempio
display(training_data['conversations'][0]['messages'])
print(100*'#')
print(tokenizer.apply_chat_template(training_data['conversations'][0]['messages'], tokenize = False, add_generation_prompt = False))

[{'role': 'system',
  'content': "Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici."},
 {'role': 'user',
  'content': "RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .\r\n\r\nIN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.\r\nADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFONODI SOSPETTI SONO

####################################################################################################
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -

In [16]:
dataset_prova = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [17]:
print(dataset_prova)
dataset_prova['conversations'][0]

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 100000
})


[{'from': 'human',
  'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
 {'from': 'gpt',
  'value': 'Boole

In [18]:
# Creiamo ora un dataset con i nostri dati
dataset = Dataset.from_dict(training_data)

In [19]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 173
})

In [20]:
dataset['conversations'][0]

{'messages': [{'content': "Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.",
   'role': 'system'},
  {'content': "RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .\r\n\r\nIN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.\r\nADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFONODI SOSPETTI SONO PR

Si nota che i due dataset sono diversi, infatti:
`dataset_prova` ha la seguente struttura:
```
{
    "conversations": [
        [
            {"role": "system", "content": <contenuto di sistema>},
            {"role": "user", "content": <contenuto di user>},
            {"role": "assistant", "content": <contenuto di assistant>}
        ],
        ...
    ]
}
```
Mentre il nostro `dataset` ha la struttura:
```
{
    "conversations": [
        {
            "messages": [
                {"role": "system", "content": <contenuto di sistema>},
                {"role": "user", "content": <contenuto di user>},
                {"role": "assistant", "content": <contenuto di assistant>}
            ]
        },
        ...
    ]
}
```
In `dataset_prova` il campo "conversations" è popolato da liste, mentre in `dataset` il campo "conversations" è popolato da dizionari, dove l'unica chiave è "messages".

Noi vogliamo aggiungere la colonna "text" al nostro `dataset` che sia di questo tipo:
```
{
    "text": [
        <testo 1>,
        <testo_2>,
        ...
    ]
}
```
cioè popolato da stringhe formattate con il chat template del modello.

In [21]:
# Creiamo la funzione da applicare successivamente al dataset
def formatting_prompts_func(data):
    conversations = data["conversations"]
    texts = [tokenizer.apply_chat_template(conversation['messages'], tokenize=False, add_generation_prompt=False) for conversation in conversations]
    return {"text" : texts}

In [22]:
# Applichiamo la funzione al datset
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

In [23]:
# Ora abbiamo creato correttamente il nuovo campo "text"
print(dataset)

Dataset({
    features: ['conversations', 'text'],
    num_rows: 173
})


In [24]:
# Osserviamo ora il primo record del dataset
display(dataset[0]['conversations'])

{'messages': [{'content': "Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.",
   'role': 'system'},
  {'content': "RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .\r\n\r\nIN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.\r\nADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFONODI SOSPETTI SONO PR

Vediamo come il chat template ha trasformato le conversazioni.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

In [25]:
print(dataset[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.
ADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOA

<a name="Train"></a>
# Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAstronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight rel

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                  Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|eot_id|>'

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.369 GB of memory reserved.
